In [1]:
import pandas as pd

df_results = pd.read_csv("data/rag-results.csv")
results = df_results.to_dict(orient="records")

In [2]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [3]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [4]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [5]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [6]:
rec = results[0]

In [7]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

In [8]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key point: late joining is allowed, and certificate eligibility depends on submitting the project before submissions close. This is semantically equivalent to the ground truth.', score='good')

In [9]:
calc_price(usage)

{'input_cost': 0.000228, 'output_cost': 0.0002295, 'total_cost': 0.0004575}

In [10]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [11]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key meaning of the ground truth: late joiners can still start, but certificate eligibility depends on submitting the project while submissions are open. This is semantically equivalent.', score='good')

In [12]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [13]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, results, judge_record)

  0%|          | 0/515 [00:00<?, ?it/s]

In [14]:
results[10]

({'question': 'How do students join the Office Hours or live workshop stream if the Zoom link is private?',
  'document': '489dd1c9d9',
  'score': 'good',
  'reasoning': 'The AI answer matches the ground truth: it correctly states that students do not use the private Zoom link, that they watch via YouTube Live, ask questions via Slido with the link pinned in chat, and that the stream URL is posted in Telegram/Slack announcements before the session. It also includes the warning not to post questions in chat. Semantically equivalent.'},
 ResponseUsage(input_tokens=452, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=89, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=541))

In [15]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [16]:
df_eval = pd.DataFrame(evaluations)

In [20]:
df_eval.head()

,question,document,score,reasoning
0,I just found this course late — can I still jo...,74eb249bbf,good,The AI answer preserves the key meaning of the...
1,Am I allowed to enroll if I only discovered th...,74eb249bbf,good,The AI answer preserves the key points: late e...
2,"If I join after the course has started, can I ...",74eb249bbf,good,The AI answer preserves the original meaning: ...
3,What do I need to do to qualify for the certif...,74eb249bbf,bad,The ground truth says that late joiners can st...
4,"Is it okay to start the course now, and what’s...",74eb249bbf,bad,"The ground truth says it is okay to start now,..."


In [17]:
calc_total_price(usages)

0.35291999999999996

In [ ]:
df_eval.score.value_counts()

score
good    492
bad      23
Name: count, dtype: int64

In [22]:
df_eval.score.value_counts(normalize=True)

score
good    0.95534
bad     0.04466
Name: proportion, dtype: float64

In [18]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 492/515 = 95.53%


In [19]:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
3,What do I need to do to qualify for the certif...,74eb249bbf,bad,The ground truth says that late joiners can st...
4,"Is it okay to start the course now, and what’s...",74eb249bbf,bad,"The ground truth says it is okay to start now,..."
21,Why does the leaderboard show some random name...,c2903069a0,bad,The AI answer captures the key point that the ...
27,Do certificates count for students who join la...,69d122f12e,bad,The AI answer does not answer the question and...
43,When should I expect the next run of llm-zoomc...,bd31146b0e,bad,The AI answer does not match the ground truth....


In [24]:
df_eval.to_csv("data/rag-evaluations.csv", index=False)